# Stage 3 -- Cross-Table Validation of All Six Model-Ready Tables

## Input
All six Stage 3 model-ready parquet files:
- `model_market_daily_means.parquet`
- `model_market_daily_full_moments.parquet`
- `model_market_monthly_means.parquet`
- `model_market_monthly_full_moments.parquet`
- `model_market_combined_means.parquet`
- `model_market_combined_full_moments.parquet`

## Purpose
A standalone cross-table validation notebook that loads all six model-ready tables simultaneously and runs six consistency checks across them. Unlike the per-table validation within each Stage 3 notebook, this checks relationships *between* tables -- ensuring targets are consistent, date ranges are properly aligned, and z-scored features have sensible distributions. Also provides a basic predictability sanity check via feature-target correlations.

---

## Checks

### Check 1: Shape Summary
Shape (rows × columns) and total NaN count reported for all six tables. NaN is computed over all columns except `date` (column 0 is skipped via `iloc[:, 1:]`). Expects zero NaN in all tables.

### Check 2: Daily Target Consistency
The combined means and combined full moments tables are supersets of the daily tables -- every trading day in the combined tables also appears in the daily tables, with the same `target_daily_return`. The combined table is inner-joined with the corresponding daily table on `date` and the two target columns are compared by Pearson correlation and exact equality. Expects correlation = 1.0 and exact match = True.

### Check 3: Monthly Target Consistency
On month-end dates (dates that appear in the standalone monthly means table), the `target_monthly_return` in the combined means table should exactly match the standalone monthly means table. The combined table is filtered to month-end dates, merged with the monthly table, and the two target columns are correlated. Expects correlation = 1.0.

### Check 4: Date Alignment
Four set comparisons are run:
- `daily_means` dates == `daily_full_moments` dates (should be True -- same daily spine)
- `combined_means` dates == `combined_full_moments` dates (should be True -- same combined spine)
- Combined dates ⊂ daily dates (should be True -- combined is a subset after monthly warmup trim)

### Check 5: Extreme Z-Scores
For each of the four daily/combined tables, the maximum absolute z-score across all numeric feature columns is reported, along with the column name achieving that maximum. Two threshold counts are also reported: columns where `max |z| > 10` and columns where `max |z| > 50`. Very high z-scores can indicate a factor that was near-constant during the expanding window period but then spiked, or a data error. The monthly-only tables are excluded since they have fewer rows and higher natural extremes.

### Check 6: Feature-Target Correlations (Daily Means)
Pearson correlations between each feature and `target_daily_return` are computed for the daily means table. Results sorted by absolute correlation, descending. Reported: top 10 features by |corr|, median |corr| across all features, count of features with |corr| > 0.05, count with |corr| > 0.10. This is a basic sanity check -- if the strongest correlations are implausibly high (>0.3) that would suggest look-ahead contamination; if they are all near zero the features may not be informative.

---

## Key Notes
- This notebook is diagnostic only -- no files are written.
- All six tables are loaded into memory simultaneously, which requires sufficient RAM given the full moments tables are large (~1,100+ columns × 4,300 rows).
- The predictability check in Check 6 uses the full sample correlation, which is not a valid out-of-sample signal -- it is purely a sanity check to confirm the features are not entirely orthogonal to the target.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

# Load all 6 tables
dm = pd.read_parquet(BASE / 'model_market_daily_means.parquet')
dfm = pd.read_parquet(BASE / 'model_market_daily_full_moments.parquet')
mm = pd.read_parquet(BASE / 'model_market_monthly_means.parquet')
mfm = pd.read_parquet(BASE / 'model_market_monthly_full_moments.parquet')
cm = pd.read_parquet(BASE / 'model_market_combined_means.parquet')
cfm = pd.read_parquet(BASE / 'model_market_combined_full_moments.parquet')

print("=" * 90)
print("CROSS-TABLE VALIDATION")
print("=" * 90)

# 1. Shape summary
print("\n1. SHAPES")
for name, df in [('daily_means', dm), ('daily_full_moments', dfm),
                  ('monthly_means', mm), ('monthly_full_moments', mfm),
                  ('combined_means', cm), ('combined_full_moments', cfm)]:
    print(f"  {name:<30s} {df.shape[0]:>6,} rows × {df.shape[1]:>5,} cols  "
          f"NaN: {df.iloc[:, 1:].isna().sum().sum()}")

# 2. Target consistency across daily tables
print("\n2. DAILY TARGET CONSISTENCY")
# Combined is a subset of daily — targets should match on shared dates
shared = cm.merge(dm[['date', 'target_daily_return']], on='date', suffixes=('_combined', '_daily'))
corr = shared['target_daily_return_combined'].corr(shared['target_daily_return_daily'])
exact_match = shared['target_daily_return_combined'].equals(shared['target_daily_return_daily'])
print(f"  combined vs daily means target: corr={corr:.10f}, exact_match={exact_match}")

shared2 = cfm.merge(dfm[['date', 'target_daily_return']], on='date', suffixes=('_combined', '_daily'))
corr2 = shared2['target_daily_return_combined'].corr(shared2['target_daily_return_daily'])
print(f"  combined_fm vs daily_fm target: corr={corr2:.10f}")

# 3. Monthly target consistency
print("\n3. MONTHLY TARGET CONSISTENCY")
# On month-end dates, combined monthly target should match standalone monthly target
month_ends = cm[cm['date'].isin(mm['date'])][['date', 'target_monthly_return']]
merged_monthly = month_ends.merge(mm[['date', 'target_monthly_return']], on='date', suffixes=('_combined', '_monthly'))
if len(merged_monthly) > 0:
    corr3 = merged_monthly['target_monthly_return_combined'].corr(merged_monthly['target_monthly_return_monthly'])
    print(f"  combined vs standalone monthly target: corr={corr3:.10f} ({len(merged_monthly)} shared dates)")

# 4. Date alignment
print("\n4. DATE ALIGNMENT")
dm_dates = set(dm['date'])
dfm_dates = set(dfm['date'])
cm_dates = set(cm['date'])
cfm_dates = set(cfm['date'])
print(f"  daily_means dates == daily_full_moments dates: {dm_dates == dfm_dates}")
print(f"  combined_means dates == combined_full_moments dates: {cm_dates == cfm_dates}")
print(f"  combined dates ⊂ daily dates: {cm_dates.issubset(dm_dates)}")

# 5. Feature distribution extremes (any absurd z-scores?)
print("\n5. EXTREME Z-SCORES (potential training issues)")
for name, df in [('daily_means', dm), ('daily_full_moments', dfm),
                  ('combined_means', cm), ('combined_full_moments', cfm)]:
    feat_cols = [c for c in df.columns if c not in ['date', 'target_daily_return', 'target_monthly_return']]
    numeric_df = df[feat_cols].select_dtypes(include=[np.number])
    abs_max = numeric_df.abs().max()
    worst_col = abs_max.idxmax()
    worst_val = abs_max.max()
    over_10 = (abs_max > 10).sum()
    over_50 = (abs_max > 50).sum()
    print(f"  {name:<30s} max |z|={worst_val:>8.1f} ({worst_col})  "
          f"cols with |z|>10: {over_10},  |z|>50: {over_50}")

# 6. Basic predictability check (top correlations with target)
print("\n6. TOP FEATURE-TARGET CORRELATIONS (daily means)")
feat_cols = [c for c in dm.columns if c not in ['date', 'target_daily_return', 'target_monthly_return']]
corrs = dm[feat_cols].corrwith(dm['target_daily_return']).abs().sort_values(ascending=False)
print(f"  Top 10 features by |corr| with daily target:")
for c in corrs.head(10).index:
    print(f"    {c:<45s} {corrs[c]:.4f}")
print(f"\n  Median |corr|: {corrs.median():.4f}")
print(f"  Features with |corr| > 0.05: {(corrs > 0.05).sum()}")
print(f"  Features with |corr| > 0.10: {(corrs > 0.10).sum()}")

print("\n" + "=" * 90)
print("VALIDATION COMPLETE")
print("=" * 90)

CROSS-TABLE VALIDATION

1. SHAPES
  daily_means                     4,299 rows ×   400 cols  NaN: 0
  daily_full_moments              4,299 rows × 1,147 cols  NaN: 0
  monthly_means                     205 rows ×   326 cols  NaN: 0
  monthly_full_moments              205 rows × 1,069 cols  NaN: 0
  combined_means                  4,299 rows ×   725 cols  NaN: 0
  combined_full_moments           4,299 rows × 2,215 cols  NaN: 0

2. DAILY TARGET CONSISTENCY
  combined vs daily means target: corr=1.0000000000, exact_match=True
  combined_fm vs daily_fm target: corr=1.0000000000

3. MONTHLY TARGET CONSISTENCY
  combined vs standalone monthly target: corr=1.0000000000 (144 shared dates)

4. DATE ALIGNMENT
  daily_means dates == daily_full_moments dates: True
  combined_means dates == combined_full_moments dates: True
  combined dates ⊂ daily dates: True

5. EXTREME Z-SCORES (potential training issues)
  daily_means                    max |z|=  8666.0 (dollarrealizedspread_lr_dw)  cols with |

In [4]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

BASE = Path('../../../Data/Data_Collection/Final/Stage_3_Model_Ready')

files = {
    'daily_means': 'model_market_daily_means.parquet',
    'daily_full_moments': 'model_market_daily_full_moments.parquet',
    'combined_means': 'model_market_combined_means.parquet',
    'combined_full_moments': 'model_market_combined_full_moments.parquet',
}

meta_cols = {'date', 'target_daily_return', 'target_monthly_return'}

for name, fname in files.items():
    # Read schema only (no data loaded — instant)
    schema = pq.read_schema(BASE / fname)
    all_cols = [f.name for f in schema]
    features = [c for c in all_cols if c not in meta_cols]
    
    print(f"\n{'=' * 90}")
    print(f"{name.upper()}: {len(features)} features")
    print(f"{'=' * 90}")
    for i, c in enumerate(features, 1):
        print(f"  {i:>4d}. {c}")
    print()


DAILY_MEANS: 398 features
     1. dlyretx
     2. dlyreti
     3. bid_ask_spread
     4. bs_ratio_inst50k_num
     5. bs_ratio_inst50k_vol
     6. bs_ratio_num
     7. bs_ratio_retail_num
     8. bs_ratio_retail_vol
     9. bs_ratio_vol
    10. dollarpriceimpact_lr_ave
    11. dollarpriceimpact_lr_dw
    12. dollarpriceimpact_lr_sw
    13. dollarrealizedspread_lr_ave
    14. dollarrealizedspread_lr_dw
    15. dollarrealizedspread_lr_sw
    16. effectivespread_dollar_ave
    17. effectivespread_dollar_dw
    18. effectivespread_dollar_sw
    19. effectivespread_percent_ave
    20. effectivespread_percent_dw
    21. effectivespread_percent_sw
    22. hindex
    23. ivol_q
    24. ivol_t
    25. n30_pos
    26. n5_pos
    27. n_obs
    28. percentpriceimpact_lr_ave
    29. percentpriceimpact_lr_dw
    30. percentpriceimpact_lr_sw
    31. percentrealizedspread_lr_ave
    32. percentrealizedspread_lr_dw
    33. percentrealizedspread_lr_sw
    34. quotedspread_dollar_tw
    35. quotedspread